This notebook applies segment-aware filtering to cleaned data, compares candidate filters quantitatively, selects the best one, and saves the filtered dataset for downstream windowing/features.

## 1. Setup and Imports

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from scipy import signal
from scipy.fft import fft, fftfreq

# Make project root importable whether CWD is repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.windowing import detect_continuous_segments

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis


In [2]:
# Load the cleaned dataset
cleaned_path = PROJECT_ROOT / "data" / "interim" / "cleaned" / "cleaned_dataset_Farm.csv"
output_filename = "filtered_dataset_Farm.csv"

if not cleaned_path.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {cleaned_path}\nPlease run 04_cleaning.ipynb first.")

df = pd.read_csv(cleaned_path)

# Ensure label column is properly typed
df['label'] = df['label'].astype('string').str.strip()

print(f"Loaded cleaned dataset: {cleaned_path}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Unique run_ids: {df['run_id'].nunique()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nRun IDs:")
for run_id in sorted(df['run_id'].unique()):
    n_samples = len(df[df['run_id'] == run_id])
    duration = df[df['run_id'] == run_id]['t_rel'].max() - df[df['run_id'] == run_id]['t_rel'].min()
    print(f"  - {run_id}: {n_samples} samples ({duration:.2f} seconds)")

Loaded cleaned dataset: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/interim/cleaned/cleaned_dataset_Farm.csv
Shape: (265863, 14)
Columns: ['t', 't_rel', 'run_id', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'v1', 'v2', 'label', 'net_speed', 'yaw_rate']
Unique run_ids: 5

Label distribution:
label
grass         95254
dirt_track    94061
soil          76548
Name: count, dtype: int64[pyarrow]

Run IDs:
  - Run1: 34309 samples (488.99 seconds)
  - Run2: 51637 samples (797.99 seconds)
  - Run3: 59814 samples (753.46 seconds)
  - Run4: 69447 samples (766.99 seconds)
  - Run5: 50656 samples (724.99 seconds)


In [3]:
# Define sensor columns and filtering policy
sensor_columns = {
    'accelerometer': ['ax', 'ay', 'az'],
    'gyroscope': ['gx', 'gy', 'gz'],
    'odometry': ['v1', 'v2', 'net_speed', 'yaw_rate']
}

imu_sensor_cols = sensor_columns['accelerometer'] + sensor_columns['gyroscope']
non_filtered_cols = sensor_columns['odometry']

print(f"IMU channels to filter: {imu_sensor_cols}")
print(f"Channels kept unfiltered by default: {non_filtered_cols}")

# Sampling rate
SAMPLE_RATE = 100  # Hz
SAMPLE_PERIOD = 1.0 / SAMPLE_RATE  # seconds

IMU channels to filter: ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
Channels kept unfiltered by default: ['v1', 'v2', 'net_speed', 'yaw_rate']


In [4]:
def compute_psd(signal_data, sample_rate):
    """
    Compute Power Spectral Density using FFT.
    
    Args:
        signal_data: 1D numpy array of signal values
        sample_rate: Sampling rate in Hz
        
    Returns:
        frequencies: Frequency values (Hz)
        psd: Power spectral density
    """
    n = len(signal_data)
    # Remove DC component (mean)
    signal_centered = signal_data - np.mean(signal_data)
    
    # Compute FFT
    fft_vals = fft(signal_centered)
    freqs = fftfreq(n, 1/sample_rate)
    
    # Only positive frequencies
    positive_mask = freqs > 0
    freqs = freqs[positive_mask]
    
    # Compute power spectral density
    psd = 2.0 * np.abs(fft_vals[positive_mask])**2 / n
    
    return freqs, psd

## 2. Filter Implementation

We evaluate Butterworth, Savitzky-Golay, and moving-average filters.
All filters are applied per continuous segment to avoid boundary artifacts.

In [5]:
def apply_butterworth_filter(data, cutoff_freq, sample_rate, order=4):
    """Apply Butterworth lowpass filter with safe handling for short segments."""
    data = np.asarray(data, dtype=float)
    if len(data) < 3:
        return data

    nyquist = sample_rate / 2
    normalized_cutoff = cutoff_freq / nyquist
    b, a = signal.butter(order, normalized_cutoff, btype='low', analog=False)

    # scipy.signal.filtfilt requires len(data) > padlen.
    padlen = 3 * max(len(a), len(b))
    if len(data) <= padlen:
        # Segment too short for stable zero-phase filtering; keep original samples.
        return data

    filtered = signal.filtfilt(b, a, data)
    return filtered


def apply_savgol_filter(data, window_length, polyorder=2):
    """Apply Savitzky-Golay smoothing filter."""
    data = np.asarray(data, dtype=float)
    if len(data) < 3:
        return data

    if window_length % 2 == 0:
        window_length += 1

    # Ensure valid SG params for short segments.
    window_length = min(window_length, len(data) if len(data) % 2 == 1 else len(data) - 1)
    window_length = max(window_length, polyorder + 2 + ((polyorder + 2) % 2 == 0))
    if window_length >= len(data):
        window_length = len(data) - 1 if len(data) % 2 == 0 else len(data)
    if window_length <= polyorder:
        return data

    filtered = signal.savgol_filter(data, window_length, polyorder, mode='nearest')
    return filtered


def apply_moving_average(data, window_size):
    """Apply moving average filter."""
    data = np.asarray(data, dtype=float)
    if len(data) < 3:
        return data

    window_size = max(1, min(window_size, len(data)))
    kernel = np.ones(window_size) / window_size
    filtered = np.convolve(data, kernel, mode='same')
    return filtered


def apply_filter_to_dataframe(df, filter_func, filter_params, sensor_cols, segments):
    """Apply a filter to sensor columns, respecting continuous segment boundaries."""
    df_filtered = df.copy()

    for segment in segments:
        segment_indices = segment['indices']

        for col in sensor_cols:
            data = df.loc[segment_indices, col].values
            if len(data) > 2:
                filtered = filter_func(data, **filter_params)
                df_filtered.loc[segment_indices, col] = filtered

    return df_filtered


print("Filter functions defined successfully!")
print("  - short segments are now handled safely (no filtfilt padlen crash)")
print("  - apply_filter_to_dataframe() keeps segment-based filtering")

Filter functions defined successfully!
  - short segments are now handled safely (no filtfilt padlen crash)
  - apply_filter_to_dataframe() keeps segment-based filtering


In [6]:
# Define filter configurations to test
filter_configs = [
    # Butterworth filters with varying cutoffs and orders
    {
        'name': 'Butterworth_5Hz_order2',
        'func': apply_butterworth_filter,
        'params': {'cutoff_freq': 5, 'sample_rate': SAMPLE_RATE, 'order': 2},
        'description': 'Aggressive lowpass at 5 Hz, gentle rolloff'
    },
    {
        'name': 'Butterworth_5Hz_order4',
        'func': apply_butterworth_filter,
        'params': {'cutoff_freq': 5, 'sample_rate': SAMPLE_RATE, 'order': 4},
        'description': 'Aggressive lowpass at 5 Hz, sharp rolloff'
    },
    {
        'name': 'Butterworth_10Hz_order2',
        'func': apply_butterworth_filter,
        'params': {'cutoff_freq': 10, 'sample_rate': SAMPLE_RATE, 'order': 2},
        'description': 'Moderate lowpass at 10 Hz, gentle rolloff'
    },
    {
        'name': 'Butterworth_10Hz_order4',
        'func': apply_butterworth_filter,
        'params': {'cutoff_freq': 10, 'sample_rate': SAMPLE_RATE, 'order': 4},
        'description': 'Moderate lowpass at 10 Hz, sharp rolloff'
    },
    {
        'name': 'Butterworth_20Hz_order2',
        'func': apply_butterworth_filter,
        'params': {'cutoff_freq': 20, 'sample_rate': SAMPLE_RATE, 'order': 2},
        'description': 'Mild lowpass at 20 Hz, gentle rolloff'
    },
    
    # Savitzky-Golay filters with varying window sizes
    {
        'name': 'SavGol_win11_poly2',
        'func': apply_savgol_filter,
        'params': {'window_length': 11, 'polyorder': 2},
        'description': 'SG filter, 0.11s window (11 samples), quadratic fit'
    },
    {
        'name': 'SavGol_win21_poly2',
        'func': apply_savgol_filter,
        'params': {'window_length': 21, 'polyorder': 2},
        'description': 'SG filter, 0.21s window (21 samples), quadratic fit'
    },
    {
        'name': 'SavGol_win21_poly3',
        'func': apply_savgol_filter,
        'params': {'window_length': 21, 'polyorder': 3},
        'description': 'SG filter, 0.21s window (21 samples), cubic fit'
    },
    {
        'name': 'SavGol_win51_poly2',
        'func': apply_savgol_filter,
        'params': {'window_length': 51, 'polyorder': 2},
        'description': 'SG filter, 0.51s window (51 samples), quadratic fit'
    },
    
    # Moving average filters
    {
        'name': 'MovAvg_win5',
        'func': apply_moving_average,
        'params': {'window_size': 5},
        'description': 'Moving average, 0.05s window (5 samples)'
    },
    {
        'name': 'MovAvg_win11',
        'func': apply_moving_average,
        'params': {'window_size': 11},
        'description': 'Moving average, 0.11s window (11 samples)'
    },
    {
        'name': 'MovAvg_win21',
        'func': apply_moving_average,
        'params': {'window_size': 21},
        'description': 'Moving average, 0.21s window (21 samples)'
    },
]

print(f"Defined {len(filter_configs)} filter configurations:")
for i, cfg in enumerate(filter_configs, 1):
    print(f"{i:2d}. {cfg['name']:25s} - {cfg['description']}")

Defined 12 filter configurations:
 1. Butterworth_5Hz_order2    - Aggressive lowpass at 5 Hz, gentle rolloff
 2. Butterworth_5Hz_order4    - Aggressive lowpass at 5 Hz, sharp rolloff
 3. Butterworth_10Hz_order2   - Moderate lowpass at 10 Hz, gentle rolloff
 4. Butterworth_10Hz_order4   - Moderate lowpass at 10 Hz, sharp rolloff
 5. Butterworth_20Hz_order2   - Mild lowpass at 20 Hz, gentle rolloff
 6. SavGol_win11_poly2        - SG filter, 0.11s window (11 samples), quadratic fit
 7. SavGol_win21_poly2        - SG filter, 0.21s window (21 samples), quadratic fit
 8. SavGol_win21_poly3        - SG filter, 0.21s window (21 samples), cubic fit
 9. SavGol_win51_poly2        - SG filter, 0.51s window (51 samples), quadratic fit
10. MovAvg_win5               - Moving average, 0.05s window (5 samples)
11. MovAvg_win11              - Moving average, 0.11s window (11 samples)
12. MovAvg_win21              - Moving average, 0.21s window (21 samples)


In [7]:
# Detect continuous segments first
GAP_THRESHOLD = 0.025  # seconds (2.5x sample period)
print("Detecting continuous segments in dataset...")
segments = detect_continuous_segments(df, gap_threshold=GAP_THRESHOLD)

print(f"✓ Detected {len(segments)} continuous segments")
print()

# Print segment statistics per run
print("Segment Statistics:")
print("=" * 80)
for run_id in sorted(df['run_id'].unique()):
    run_segments = [s for s in segments if s['run_id'] == run_id]
    total_samples = sum(s['n_samples'] for s in run_segments)
    print(f"  {run_id}:")
    print(f"    - {len(run_segments)} segment(s), {total_samples} total samples")

    if len(run_segments) > 1:
        for seg in run_segments[:18]:
            duration = seg['end_time'] - seg['start_time']
            print(f"      • Segment {seg['segment_id']}: {seg['n_samples']} samples, "
                  f"{duration:.2f}s ({seg['start_time']:.2f} - {seg['end_time']:.2f})")
        if len(run_segments) > 18:
            print("      • ... (truncated)")
print()

segment_summary = pd.DataFrame(segments)[['run_id', 'segment_id', 'n_samples', 'start_time', 'end_time']].copy()
segment_summary['duration_s'] = segment_summary['end_time'] - segment_summary['start_time']
segment_summary = segment_summary.sort_values(['run_id', 'segment_id']).reset_index(drop=True)

print("Segment summary table (first 20 rows):")
print(segment_summary.head(20).to_string(index=False))
print()

# Explicitly skip tiny segments for filtering
MIN_SEGMENT_SAMPLES_FOR_FILTER = 15  # ~0.15s at 100 Hz
filterable_segments = [s for s in segments if s['n_samples'] >= MIN_SEGMENT_SAMPLES_FOR_FILTER]
skipped_segments = [s for s in segments if s['n_samples'] < MIN_SEGMENT_SAMPLES_FOR_FILTER]

n_zero_duration = int((segment_summary['duration_s'] <= 0.0).sum())
print("Filtering eligibility:")
print(f"  - Minimum segment samples for filtering: {MIN_SEGMENT_SAMPLES_FOR_FILTER}")
print(f"  - Filterable segments: {len(filterable_segments)}")
print(f"  - Skipped tiny segments: {len(skipped_segments)}")
print(f"  - Zero-duration segments detected: {n_zero_duration}")
print()

# Apply all filters with segment-aware filtering
print("Applying filters to dataset (segment-aware)...")
print(f"Original data shape: {df.shape}")
print(f"Filtering IMU channels only: {imu_sensor_cols}")
print(f"Preserving unfiltered channels: {non_filtered_cols}")
print(f"Filtering {len(imu_sensor_cols)} sensor columns across {len(filterable_segments)} eligible segments")
print()

filtered_datasets = {}

for cfg in filter_configs:
    print(f"Applying {cfg['name']}...", end=' ')

    df_filtered = apply_filter_to_dataframe(
        df,
        cfg['func'],
        cfg['params'],
        imu_sensor_cols,
        filterable_segments
    )

    filtered_datasets[cfg['name']] = df_filtered
    print("✓")

print(f"\n✓ Successfully created {len(filtered_datasets)} filtered datasets!")
print("  All filters applied with gap-aware processing")

Detecting continuous segments in dataset...
✓ Detected 2196 continuous segments

Segment Statistics:
  Run1:
    - 295 segment(s), 34309 total samples
      • Segment 0: 595 samples, 5.94s (28.10 - 34.04)
      • Segment 1: 1 samples, 0.00s (34.07 - 34.07)
      • Segment 2: 72 samples, 0.72s (34.15 - 34.87)
      • Segment 3: 2 samples, 0.01s (34.90 - 34.91)
      • Segment 4: 1286 samples, 12.91s (34.94 - 47.85)
      • Segment 5: 659 samples, 6.60s (47.88 - 54.48)
      • Segment 6: 1353 samples, 13.55s (54.54 - 68.09)
      • Segment 7: 85 samples, 0.85s (85.37 - 86.22)
      • Segment 8: 742 samples, 7.44s (86.25 - 93.69)
      • Segment 9: 147 samples, 1.47s (93.72 - 95.19)
      • Segment 10: 17 samples, 0.18s (95.27 - 95.45)
      • Segment 11: 111 samples, 1.12s (95.51 - 96.63)
      • Segment 12: 175 samples, 1.78s (96.66 - 98.44)
      • Segment 13: 1 samples, 0.00s (98.47 - 98.47)
      • Segment 14: 12 samples, 0.11s (98.51 - 98.62)
      • Segment 15: 10 samples, 0.09s (9

## 3. Quantitative Filter Comparison

Compute compact metrics to compare candidate filters:
1. Correlation with raw signal (preservation)
2. Smoothness improvement
3. High-frequency noise reduction
4. RMS residual

In [8]:
def compute_filter_metrics(raw_signal, filtered_signal, sample_rate):
    """
    Compute quality metrics for a filtered signal.

    Returns:
        dict with metrics
    """
    # 1. Correlation with raw signal
    correlation = np.corrcoef(raw_signal, filtered_signal)[0, 1]

    # 2. Smoothness (lower derivative std = smoother)
    raw_derivative = np.diff(raw_signal)
    filtered_derivative = np.diff(filtered_signal)
    smoothness_improvement = np.std(raw_derivative) / np.std(filtered_derivative)

    # 3. High-frequency noise reduction
    # Estimate noise as high-frequency component
    freqs, psd_raw = compute_psd(raw_signal, sample_rate)
    _, psd_filtered = compute_psd(filtered_signal, sample_rate)

    # Noise is power above 15 Hz
    high_freq_mask = freqs > 15
    if np.sum(high_freq_mask) > 0:
        noise_raw = np.sum(psd_raw[high_freq_mask])
        noise_filtered = np.sum(psd_filtered[high_freq_mask])
        noise_reduction = noise_raw / (noise_filtered + 1e-10)  # Avoid division by zero
    else:
        noise_reduction = 1.0

    # 4. RMS of residual (difference between raw and filtered)
    residual = raw_signal - filtered_signal
    rms_residual = np.sqrt(np.mean(residual**2))

    return {
        'correlation': correlation,
        'smoothness_improvement': smoothness_improvement,
        'noise_reduction': noise_reduction,
        'rms_residual': rms_residual
    }


# Compute metrics for all filters (IMU channels only)
print("Computing metrics for all filters (IMU channels only)...")
metrics_results = []
metric_channels = imu_sensor_cols

for filter_name, df_filtered in filtered_datasets.items():
    filter_metrics = {'filter_name': filter_name}

    # Average metrics across filtered IMU channels
    for sensor_col in metric_channels:
        raw_signal = df[sensor_col].values
        filtered_signal = df_filtered[sensor_col].values

        metrics = compute_filter_metrics(raw_signal, filtered_signal, SAMPLE_RATE)

        for metric_name, value in metrics.items():
            key = f'{sensor_col}_{metric_name}'
            filter_metrics[key] = value

    # Compute average across filtered channels
    filter_metrics['avg_correlation'] = np.mean([filter_metrics[f'{col}_correlation'] for col in metric_channels])
    filter_metrics['avg_smoothness_improvement'] = np.mean([filter_metrics[f'{col}_smoothness_improvement'] for col in metric_channels])
    filter_metrics['avg_noise_reduction'] = np.mean([filter_metrics[f'{col}_noise_reduction'] for col in metric_channels])
    filter_metrics['avg_rms_residual'] = np.mean([filter_metrics[f'{col}_rms_residual'] for col in metric_channels])

    metrics_results.append(filter_metrics)

metrics_df = pd.DataFrame(metrics_results)
print("✓ Metrics computed!")
print(f"  Channels used for ranking: {metric_channels}")

Computing metrics for all filters (IMU channels only)...
✓ Metrics computed!
  Channels used for ranking: ['ax', 'ay', 'az', 'gx', 'gy', 'gz']


In [9]:
# Display metrics summary
summary_cols = ['filter_name', 'avg_correlation', 'avg_smoothness_improvement', 
                'avg_noise_reduction', 'avg_rms_residual']
metrics_summary = metrics_df[summary_cols].copy()

# Sort by different criteria
print("=" * 80)
print("FILTER PERFORMANCE METRICS")
print("=" * 80)
print()

print("📊 All Filters - Summary Metrics")
print("-" * 80)
print(metrics_summary.to_string(index=False))
print()

print("🏆 Top 5 by Signal Preservation (highest correlation):")
print(metrics_summary.nlargest(5, 'avg_correlation')[['filter_name', 'avg_correlation']].to_string(index=False))
print()

print("🏆 Top 5 by Smoothness (highest smoothness improvement):")
print(metrics_summary.nlargest(5, 'avg_smoothness_improvement')[['filter_name', 'avg_smoothness_improvement']].to_string(index=False))
print()

print("🏆 Top 5 by Noise Reduction (highest noise reduction):")
print(metrics_summary.nlargest(5, 'avg_noise_reduction')[['filter_name', 'avg_noise_reduction']].to_string(index=False))
print()

print("🏆 Top 5 by Low Residual (smallest difference from raw):")
print(metrics_summary.nsmallest(5, 'avg_rms_residual')[['filter_name', 'avg_rms_residual']].to_string(index=False))

FILTER PERFORMANCE METRICS

📊 All Filters - Summary Metrics
--------------------------------------------------------------------------------
            filter_name  avg_correlation  avg_smoothness_improvement  avg_noise_reduction  avg_rms_residual
 Butterworth_5Hz_order2         0.737479                    4.021570            28.610511          1.250717
 Butterworth_5Hz_order4         0.705197                    3.918357            27.383093          1.294982
Butterworth_10Hz_order2         0.831642                    3.110994            26.902302          0.911367
Butterworth_10Hz_order4         0.820095                    3.002479            26.799155          0.932992
Butterworth_20Hz_order2         0.912309                    2.157153             7.773444          0.639438
     SavGol_win11_poly2         0.808556                    2.813060            20.276301          0.944452
     SavGol_win21_poly2         0.746754                    3.777918            26.369510          1.20

## 4. Filter Selection

Select the final filter using quantitative thresholds and a composite score.

In [10]:
# Apply selection criteria
good_filters = metrics_summary[
    (metrics_summary['avg_correlation'] > 0.90) &
    (metrics_summary['avg_noise_reduction'] > 2.0)
].copy()

print("Filters meeting quality criteria (correlation > 0.90, noise reduction > 2.0):")
print("=" * 80)
print(good_filters.to_string(index=False))
print()

# Rank by balanced score
# Score = correlation * noise_reduction * smoothness_improvement / rms_residual
good_filters['composite_score'] = (
    good_filters['avg_correlation'] * 
    good_filters['avg_noise_reduction'] * 
    good_filters['avg_smoothness_improvement'] / 
    (good_filters['avg_rms_residual'] + 0.01)  # Avoid division by zero
)

good_filters = good_filters.sort_values('composite_score', ascending=False)

print("\n🏆 Top Recommended Filters (by composite score):")
print("=" * 80)
print(good_filters[['filter_name', 'composite_score', 'avg_correlation', 
                     'avg_noise_reduction']].to_string(index=False))

if len(good_filters) > 0:
    best_filter_name = good_filters.iloc[0]['filter_name']
    print(f"\n✅ RECOMMENDED FILTER: {best_filter_name}")
    print()
    
    # Get filter config
    best_config = next(cfg for cfg in filter_configs if cfg['name'] == best_filter_name)
    print(f"Description: {best_config['description']}")
    print(f"Parameters: {best_config['params']}")
else:
    print("\n⚠️  No filters met the strict criteria. Consider relaxing thresholds or choosing manually.")
    best_filter_name = metrics_summary.nlargest(1, 'avg_correlation').iloc[0]['filter_name']
    print(f"Fallback recommendation (best correlation): {best_filter_name}")

Filters meeting quality criteria (correlation > 0.90, noise reduction > 2.0):
            filter_name  avg_correlation  avg_smoothness_improvement  avg_noise_reduction  avg_rms_residual
Butterworth_20Hz_order2         0.912309                    2.157153             7.773444          0.639438


🏆 Top Recommended Filters (by composite score):
            filter_name  composite_score  avg_correlation  avg_noise_reduction
Butterworth_20Hz_order2        23.555865         0.912309             7.773444

✅ RECOMMENDED FILTER: Butterworth_20Hz_order2

Description: Mild lowpass at 20 Hz, gentle rolloff
Parameters: {'cutoff_freq': 20, 'sample_rate': 100, 'order': 2}


## 5. Save Filtered Dataset

Save the selected filtered dataset and metadata for downstream notebooks (05_windowing, 06_features, 07_models).

In [11]:
# Select which filter to save (change this after reviewing results)
filter_to_save = best_filter_name  # Or manually specify: 'Butterworth_10Hz_order2'

print(f"Saving filtered dataset: {filter_to_save}")
print()

# Get the filtered dataset
df_to_save = filtered_datasets[filter_to_save].copy()

# Create output directory
output_dir = PROJECT_ROOT / "data" / "interim" / "filtered"
output_dir.mkdir(parents=True, exist_ok=True)

# Save as CSV
output_path = output_dir / output_filename
df_to_save.to_csv(output_path, index=False)

print(f"✓ Saved filtered dataset to: {output_path}")
print(f"  Shape: {df_to_save.shape}")
print(f"  Columns: {list(df_to_save.columns)}")
print()

# Save filter metadata
metadata = {
    'filter_name': filter_to_save,
    'filter_config': next(cfg for cfg in filter_configs if cfg['name'] == filter_to_save),
    'source_file': str(cleaned_path),
    'output_file': str(output_path),
    'num_samples': len(df_to_save),
    'num_runs': df_to_save['run_id'].nunique(),
    'sampling_rate_hz': SAMPLE_RATE,
    'filtered_columns': imu_sensor_cols,
    'preserved_unfiltered_columns': non_filtered_cols,
    'metrics': metrics_summary[metrics_summary['filter_name'] == filter_to_save].iloc[0].to_dict()
}

import json
metadata_path = output_dir / "filter_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"✓ Saved filter metadata to: {metadata_path}")
print()
print("=" * 80)
print("FILTERING COMPLETE!")
print("=" * 80)
print()
print("Next steps:")
print("  1. Review the visualizations and metrics above")
print("  2. If you want a different filter, change 'filter_to_save' and re-run this cell")
print("  3. Proceed to 05_windowing.ipynb to create fixed-length windows")
print("  4. The filtered dataset is ready at: data/interim/filtered/filtered_dataset.csv")

Saving filtered dataset: Butterworth_20Hz_order2

✓ Saved filtered dataset to: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/interim/filtered/filtered_dataset_Farm.csv
  Shape: (265863, 14)
  Columns: ['t', 't_rel', 'run_id', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'v1', 'v2', 'label', 'net_speed', 'yaw_rate']

✓ Saved filter metadata to: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/interim/filtered/filter_metadata.json

FILTERING COMPLETE!

Next steps:
  1. Review the visualizations and metrics above
  2. If you want a different filter, change 'filter_to_save' and re-run this cell
  3. Proceed to 05_windowing.ipynb to create fixed-length windows
  4. The filtered dataset is ready at: data/interim/filtered/filtered_dataset.csv
